In [25]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

In [26]:
# CSV vaddu - direct data generate
df = pd.DataFrame({
    'tenure': np.random.randint(1,72,5000),
    'monthly_charges': np.random.uniform(20,120,5000),
    'support_tickets': np.random.randint(0,10,5000),
    'contract_monthly': np.random.choice([0,1],5000),
    'total_charges': np.random.uniform(100,8000,5000),
    'churn': np.random.choice([0,1],5000, p=[0.73,0.27])
})
df.head()

,tenure,monthly_charges,support_tickets,contract_monthly,total_charges,churn
0,50,42.676173,5,0,6345.685868,0
1,2,73.070147,8,1,6000.705784,0
2,27,101.769386,4,0,1540.618554,1
3,8,104.010980,7,0,936.945092,0
4,48,58.509068,5,0,4992.220746,1


In [27]:
X = df[['tenure','monthly_charges','support_tickets','contract_monthly','total_charges']]
y = df['churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

# 1. Data
df = pd.DataFrame({
    'tenure': np.random.randint(1,72,5000),
    'monthly_charges': np.random.uniform(20,120,5000),
    'support_tickets': np.random.randint(0,10,5000),
    'contract_monthly': np.random.choice([0,1],5000),
    'total_charges': np.random.uniform(100,8000,5000),
    'churn': np.random.choice([0,1],5000, p=[0.73,0.27])
})

# 2. Model
X = df[['tenure','monthly_charges','support_tickets','contract_monthly','total_charges']]
y = df['churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
model = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, eval_metric='logloss')
model.fit(X_train_s, y_train)

# 3. Churn Risk
df['churn_risk'] = model.predict_proba(scaler.transform(X))[:,1] * 100

# 4. Retention
def retention_engine(row):
    if row['churn_risk'] < 70: return "NO_ACTION"
    if row['monthly_charges'] > 85: return "DISCOUNT_25_PCT"
    if row['support_tickets'] >= 5: return "PRIORITY_SUPPORT_CALL"
    if row['tenure'] <= 6: return "ONBOARDING_ASSIST"
    return "RETENTION_OFFER"

df['retention_action'] = df.apply(retention_engine, axis=1)

# 5. Kaplan Meier
def kaplan_meier(data, duration_col='tenure', event_col='churn'):
    data_sorted = data.sort_values(duration_col)
    durations = sorted(data_sorted[duration_col].unique())
    at_risk = len(data_sorted)
    surv = 1.0
    result = []
    for t in durations:
        died = ((data_sorted[duration_col]==t) & (data_sorted[event_col]==1)).sum()
        if at_risk > 0: surv = surv * (1 - died/at_risk)
        result.append((t, surv))
        at_risk -= (data_sorted[duration_col]==t).sum()
    return pd.DataFrame(result, columns=['duration','survival_prob'])

km_df = kaplan_meier(df, 'tenure', 'churn')

# 6. Final
final_output = df[['tenure','monthly_charges','support_tickets','churn_risk','retention_action']].sort_values(by='churn_risk', ascending=False)
print(km_df.head())
print(final_output.head(20))

   duration  survival_prob
0         1       0.997200
1         2       0.992748
2         3       0.989471
3         4       0.985747
4         5       0.980732
      tenure  monthly_charges  support_tickets  churn_risk  \
2982       4        66.523485                0   89.952736   
3133       3        66.243157                0   89.032059   
4959       2       119.744576                9   85.985054   
869       23        32.797743                7   83.940140   
3283       2       119.684007                9   82.258247   
4902      67       113.623720                8   82.141975   
3722       7       119.728380                0   80.322144   
3867      67       119.619898                8   78.828445   
3923      39        22.556996                4   78.100517   
1668      52        20.706155                9   76.950111   
4104      62        49.251222                5   75.696739   
1599       5        73.297888                1   75.283806   
2826      21       119.533217   

In [29]:
importance = model.feature_importances_
shap_like_df = pd.DataFrame({'feature': X.columns, 'impact': importance}).sort_values('impact', ascending=False)
shap_like_df

,feature,impact
3,contract_monthly,0.217980
4,total_charges,0.201627
1,monthly_charges,0.197613
0,tenure,0.193903
2,support_tickets,0.188877


In [30]:
def retention_engine(row):
    if row['churn_risk'] < 70:
        return "NO_ACTION"
    if row['monthly_charges'] > 85:
        return "DISCOUNT_25_PCT"
    if row['support_tickets'] >= 5:
        return "PRIORITY_SUPPORT_CALL"
    if row['tenure'] <= 6:
        return "ONBOARDING_ASSIST"
    return "RETENTION_OFFER"

df['retention_action'] = df.apply(retention_engine, axis=1)
final_output = df[['tenure','monthly_charges','support_tickets','churn_risk','retention_action']].sort_values(by='churn_risk', ascending=False)
final_output.head(20)

,tenure,monthly_charges,support_tickets,churn_risk,retention_action
2982,4,66.523485,0,89.952736,ONBOARDING_ASSIST
3133,3,66.243157,0,89.032059,ONBOARDING_ASSIST
4959,2,119.744576,9,85.985054,DISCOUNT_25_PCT
869,23,32.797743,7,83.940140,PRIORITY_SUPPORT_CALL
3283,2,119.684007,9,82.258247,DISCOUNT_25_PCT
4902,67,113.623720,8,82.141975,DISCOUNT_25_PCT
3722,7,119.728380,0,80.322144,DISCOUNT_25_PCT
3867,67,119.619898,8,78.828445,DISCOUNT_25_PCT
3923,39,22.556996,4,78.100517,RETENTION_OFFER
1668,52,20.706155,9,76.950111,PRIORITY_SUPPORT_CALL


In [31]:
importance = model.feature_importances_
shap_like_df = pd.DataFrame({'feature': X.columns, 'impact': importance}).sort_values('impact', ascending=False)
shap_like_df

,feature,impact
3,contract_monthly,0.217980
4,total_charges,0.201627
1,monthly_charges,0.197613
0,tenure,0.193903
2,support_tickets,0.188877


In [32]:
def retention_engine(row):
    if row['churn_risk'] < 70:
        return "NO_ACTION"
    if row['monthly_charges'] > 85:
        return "DISCOUNT_25_PCT"
    if row['support_tickets'] >= 5:
        return "PRIORITY_SUPPORT_CALL"
    if row['tenure'] <= 6:
        return "ONBOARDING_ASSIST"
    return "RETENTION_OFFER"

df['retention_action'] = df.apply(retention_engine, axis=1)
df[['churn_risk','retention_action']].to_csv('retention_output.csv', index=False)

In [33]:
final_output = df[['tenure','monthly_charges','support_tickets','churn_risk','retention_action']].sort_values(by='churn_risk', ascending=False)
final_output.head(20)

,tenure,monthly_charges,support_tickets,churn_risk,retention_action
2982,4,66.523485,0,89.952736,ONBOARDING_ASSIST
3133,3,66.243157,0,89.032059,ONBOARDING_ASSIST
4959,2,119.744576,9,85.985054,DISCOUNT_25_PCT
869,23,32.797743,7,83.940140,PRIORITY_SUPPORT_CALL
3283,2,119.684007,9,82.258247,DISCOUNT_25_PCT
4902,67,113.623720,8,82.141975,DISCOUNT_25_PCT
3722,7,119.728380,0,80.322144,DISCOUNT_25_PCT
3867,67,119.619898,8,78.828445,DISCOUNT_25_PCT
3923,39,22.556996,4,78.100517,RETENTION_OFFER
1668,52,20.706155,9,76.950111,PRIORITY_SUPPORT_CALL
